# Fine-Tuning FinBERT for Financial Sentiment Classification

## 1. Project Overview


### 1.1 Problem Statement
Financial markets generate thousands of news sentences daily. Manually
categorizing each sentence as positive, negative, or neutral is not
scalable. This project builds an automated sentiment classifier that
reads financial news sentences and predicts their market sentiment
in real time.

### 1.2 Business Use Case
- Algorithmic trading: generate buy/sell signals from news sentiment
- Risk monitoring: automatically flag negative news about portfolio companies
- Earnings call analysis: gauge executive tone in real time
- Investment research: summarize sentiment across thousands of analyst reports

### 1.3 Approach
Fine-tuned ProsusAI/FinBERT, a BERT model pre-trained on financial text -
on the Financial PhraseBank dataset (sentences_75agree, 3,453 sentences)
for 3-class sentiment classification (negative, neutral, positive).

Pipeline: Data loading → Stratified splitting → Tokenization →
Fine-tuning → Evaluation → Inference

## 2. Environment Setup


### 2.1 GPU Check

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


### 2.2 Library Installation

In [2]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


### 2.3 Imports

In [3]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Transformers ready: {True}")
print(f"Datasets ready: {True}")

Using device: cuda
Transformers ready: True
Datasets ready: True


## 3. Dataset


### 3.1 Load Dataset

In [5]:
import requests, zipfile, io, pandas as pd
from datasets import Dataset

# Step 1: Download the original dataset zip from GitHub
url = "https://github.com/neoyipeng2018/FinancialPhraseBank-v1.0/raw/main/FinancialPhraseBank-v1.0.zip"
r = requests.get(url)

# Step 2: Extract the zip file in memory
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(".")

# Step 3: Read the 75agree text file into a dataframe
rows = []
with open("FinancialPhraseBank-v1.0/Sentences_75Agree.txt", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if "@" in line:
            sentence, label = line.rsplit("@", 1)
            rows.append({"sentence": sentence.strip(), "label": label.strip()})

df = pd.DataFrame(rows)

# Step 4: Convert string labels to numbers
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label"] = df["label"].map(label_map)

print(df.shape)
print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())

(3453, 2)
                                            sentence  label
0  According to Gran , the company has no plans t...      1
1  With the new production plant the company woul...      2
2  For the last quarter of 2010 , Componenta 's n...      2
3  In the third quarter of 2010 , net sales incre...      2
4  Operating profit rose to EUR 13.1 mn from EUR ...      2

Label distribution:
label
1    2146
2     887
0     420
Name: count, dtype: int64


### 3.2 Stratified Spitting of data

In [6]:
from sklearn.model_selection import train_test_split

# Step 1: First split — separate train (70%) from temp (30%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

# Step 2: Second split — split temp (30%) into val (15%) and test (15%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

# Verify shapes
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Verify stratification worked
print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True).round(2))

print("\nVal label distribution:")
print(val_df["label"].value_counts(normalize=True).round(2))

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True).round(2))

Train: 2417 | Val: 518 | Test: 518

Train label distribution:
label
1    0.62
2    0.26
0    0.12
Name: proportion, dtype: float64

Val label distribution:
label
1    0.62
2    0.26
0    0.12
Name: proportion, dtype: float64

Test label distribution:
label
1    0.62
2    0.26
0    0.12
Name: proportion, dtype: float64


In [7]:
# I want to check sentence lengths to confirm max_length=128 is safe for my dataset

# I calculate word count per sentence as a rough estimate of token count
# actual token count is slightly higher due to subword splitting, but this gives a good estimate
sentence_lengths = df["sentence"].apply(lambda x: len(x.split()))

print(f"Average length:        {sentence_lengths.mean():.1f} words")
print(f"Max length:            {sentence_lengths.max()} words")
print(f"95th percentile:       {sentence_lengths.quantile(0.95):.1f} words")
print(f"99th percentile:       {sentence_lengths.quantile(0.99):.1f} words")

Average length:        22.8 words
Max length:            81 words
95th percentile:       42.4 words
99th percentile:       50.0 words


### 3.3 Preprocessing & Tokenization

In [8]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# loaded the FinBERT tokenizer which knows the exact vocabulary FinBERT was trained on
# tokenizer is like a real dictionary/encyclopedia here which knows all words
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

# converted my pandas dataframes into HuggingFace Dataset objects
# Trainer only works with HuggingFace Datasets, not pandas dataframes
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset   = Dataset.from_pandas(val_df,   preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df,  preserve_index=False)

# bundled all three splits into one DatasetDict so I can process them together
dataset = DatasetDict({
    "train":      train_dataset,
    "validation": val_dataset,
    "test":       test_dataset
})

# defined a tokenization function that will be applied to every row
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# applyed the tokenization function to all three splits at once
tokenized_dataset = dataset.map(tokenize, batched=True)

print(tokenized_dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/2417 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/518 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2417
    })
    validation: Dataset({
        features: ['sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 518
    })
    test: Dataset({
        features: ['sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 518
    })
})


In [9]:
# Printing to see what tokenization actually did to a real sentence
example = tokenized_dataset["train"][0]

print("Original sentence:")
print(example["sentence"])

print("\nLabel:", example["label"])

print("\nFirst 20 token IDs:")
print(example["input_ids"][:20])

print("\nFirst 20 attention mask values:")
print(example["attention_mask"][:20])

print("\nTokens decoded back to words:")
print(tokenizer.convert_ids_to_tokens(example["input_ids"][:20]))

Original sentence:
Also , a seven-year historic analysis is provided for these markets .

Label: 1

First 20 token IDs:
[101, 2036, 1010, 1037, 2698, 1011, 2095, 3181, 4106, 2003, 3024, 2005, 2122, 6089, 1012, 102, 0, 0, 0, 0]

First 20 attention mask values:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]

Tokens decoded back to words:
['[CLS]', 'also', ',', 'a', 'seven', '-', 'year', 'historic', 'analysis', 'is', 'provided', 'for', 'these', 'markets', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


## 4. Model

### 4.1 Load Pre-trained FinBERT

In [10]:
from transformers import AutoModelForSequenceClassification
import torch

# defined label mappings so the model knows what each output class means
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

# loaded FinBERT with a classification head on top for 3 output classes
model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# moved model to GPU so training runs on CUDA instead of CPU
model = model.to(device)

#numel is number of elements
print(f"Model loaded on: {next(model.parameters()).device}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda:0
Number of parameters: 109,484,547


### 4.2 Fine-Tuning Configuration



- **Base model:** ProsusAI/finbert
- **Task:** Sequence Classification (3 classes)
- **Classes:** Negative (0), Neutral (1), Positive (2)
- **Parameters:** 109M
- **Device:** CUDA (T4 GPU)
- **Max sequence length:** 128 tokens

## 5. Training

### 5.1 Define Metrics

In [11]:
import evaluate
import numpy as np

# loaded accuracy and f1 metrics from HuggingFace evaluate library
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

# defined function that Trainer calls after each epoch to evaluate performance
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # argmax converts raw logits to predicted class by taking highest score
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)

    # average="weighted" means F1 averaged across all 3 classes
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

### 5.2 Train Model

In [12]:
from transformers import TrainingArguments, Trainer

# defined all training hyperparameters
training_args = TrainingArguments(
    output_dir="./finbert-financial-sentiment",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
)

# assembled Trainer with model, data, and configuration
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

# started fine-tuning
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.316996,0.245404,0.913127,0.914660
2,0.153478,0.236642,0.934363,0.934495
3,0.040732,0.242725,0.945946,0.946687
4,0.020919,0.233004,0.940154,0.940194
5,0.005223,0.238639,0.940154,0.940470


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=760, training_loss=0.27131601123041227, metrics={'train_runtime': 346.0926, 'train_samples_per_second': 34.918, 'train_steps_per_second': 2.196, 'total_flos': 794931413310720.0, 'train_loss': 0.27131601123041227, 'epoch': 5.0})

## 6. Evaluation

### 6.1 Results

In [13]:
# evaluated model on test set for the first time
test_results = trainer.evaluate(tokenized_dataset["test"])

print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test F1:       {test_results['eval_f1']:.4f}")

Test Accuracy: 0.9653
Test F1:       0.9651


In [14]:
from sklearn.metrics import classification_report
import numpy as np

# ran model on test set to get raw predictions
predictions = trainer.predict(tokenized_dataset["test"])

# converted raw logits to predicted class labels
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# printed per class breakdown so we can see performance on each sentiment
print(classification_report(
    true_labels,
    predicted_labels,
    target_names=["negative", "neutral", "positive"]
))

              precision    recall  f1-score   support

    negative       1.00      0.87      0.93        63
     neutral       0.98      0.98      0.98       322
    positive       0.93      0.97      0.95       133

    accuracy                           0.97       518
   macro avg       0.97      0.94      0.95       518
weighted avg       0.97      0.97      0.97       518



## 7. Inference

### 7.1 Predict on Custom Sentences

In [15]:
import torch

# mapped numeric predictions back to human readable labels
id2label = {0: "negative", 1: "neutral", 2: "positive"}

def predict_sentiment(text):
    # tokenized the input sentence the same way training data was tokenized
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # moved input tensors to GPU to match where model lives
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # disabled gradient calculation since we are only predicting, not training
    with torch.no_grad():
        outputs = model(**inputs)

    # converted raw logits to probabilities using softmax
    probs = torch.softmax(outputs.logits, dim=1).squeeze()

    # picked the class with highest probability as final prediction
    predicted_class = torch.argmax(probs).item()
    predicted_label = id2label[predicted_class]

    return {
        "sentence": text,
        "prediction": predicted_label,
        "confidence": {
            id2label[i]: round(probs[i].item() * 100, 2)
            for i in range(3)
        }
    }

In [16]:
test_sentences = [
    "The company reported record profits for the third consecutive quarter.",
    "Operating losses widened significantly amid declining demand.",
    "The firm operates across 12 countries in the European market.",
    "Revenues fell sharply as supply chain disruptions continued.",
    "Nokia announced a strategic partnership with Microsoft to expand cloud services."
]

for sentence in test_sentences:
    result = predict_sentiment(sentence)
    print(f"Sentence:   {result['sentence']}")
    print(f"Prediction: {result['prediction'].upper()}")
    print(f"Confidence: {result['confidence']}")
    print("-" * 80)

Sentence:   The company reported record profits for the third consecutive quarter.
Prediction: POSITIVE
Confidence: {'negative': 0.26, 'neutral': 0.11, 'positive': 99.63}
--------------------------------------------------------------------------------
Sentence:   Operating losses widened significantly amid declining demand.
Prediction: NEGATIVE
Confidence: {'negative': 98.36, 'neutral': 0.9, 'positive': 0.73}
--------------------------------------------------------------------------------
Sentence:   The firm operates across 12 countries in the European market.
Prediction: NEUTRAL
Confidence: {'negative': 0.07, 'neutral': 99.8, 'positive': 0.13}
--------------------------------------------------------------------------------
Sentence:   Revenues fell sharply as supply chain disruptions continued.
Prediction: NEGATIVE
Confidence: {'negative': 99.09, 'neutral': 0.68, 'positive': 0.23}
--------------------------------------------------------------------------------
Sentence:   Nokia annou

## 8. Conclusion & Key Takeaways

### What was built
An end-to-end financial sentiment classification pipeline using FinBERT,
a domain-specific transformer pre-trained on financial text. The model
classifies financial news sentences into three categories:
negative, neutral, and positive.

### Results
- Test Accuracy: 96.5%
- Test F1 Score: 96.5%
- Negative class precision: 100% (zero false positives)

### Technical skills demonstrated
- Transformer fine-tuning using HuggingFace Transformers and PyTorch
- Stratified train/validation/test splitting to handle class imbalance
- Tokenization with padding, truncation and attention masking
- GPU-accelerated training on Google Colab (T4)
- Per-class evaluation using precision, recall and F1

### Dataset
Financial PhraseBank (sentences_75agree) - 3,453 expert-annotated
financial news sentences with 75%+ annotator agreement on labels.

### Summary
Raw text file → parsed dataset → stratified splits → tokenization → FinBERT fine-tuning → evaluation → inference function

In [17]:
import subprocess
subprocess.run(["pip", "install", "nbformat", "-q"])

import nbformat

# read the current notebook
with open("/content/Fine_Tuning_FinBERT_for_Financial_Sentiment.ipynb", "r") as f:
    nb = nbformat.read(f, as_version=4)

# removed widget metadata that breaks GitHub rendering
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

# saved cleaned version
with open("/content/Fine_Tuning_FinBERT_for_Financial_Sentiment_clean.ipynb", "w") as f:
    nbformat.write(nb, f)

print("Done")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Fine_Tuning_FinBERT_for_Financial_Sentiment.ipynb'

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os

# searched for the notebook file in Google Drive
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))

/content/drive/MyDrive/Colab Notebooks/Fine_Tuning_FinBERT_for_Financial_Sentiment.ipynb
/content/drive/MyDrive/Colab Notebooks/Demo_FA25.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of FA25_Lecture3_Data_Types_Functions.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of FA25_Lecture4_Functions.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture5_Lists_If_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture4_Functions_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture6_If_For_Loops_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture7_For_Loops_2DLists_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture8_2DLists_Dictionary_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture9_Dictionary_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/FA25_Assignment1_Python_Fundamentals_Bawa.ipynb
/content/drive/MyDrive/Colab Notebooks/PA_FA25_Lecture11_While_FileHandling_Bawa.ipynb
/content/drive/MyDrive/Colab Notebo